## Aggregate LOOCV Results
This script searches through all subfolders of a given input folder (except those named "Archive"), 
finds all files named `loocv_results.csv`, extracts the first row of each file keeping only the columns:
- `stats.bias`
- `stats.MSPE`
- `stats.RMSPE`
- `stats.RAV`


In [1]:

import os
import pandas as pd

# Set the path to your input folder (adjust as needed)

input_folders = ['CPF', 'ETF'] 
output_file = 'Results Summary\loocv_summary.csv'

# List to collect summary rows
summary_rows = []

for input_folder in input_folders:
    # Walk through the input folder recursively
    for root, dirs, files in os.walk(input_folder):
        # Skip any subfolder named "Archive" (case insensitive)
        dirs[:] = [d for d in dirs if d.lower() != "archive"]
        
        if "loocv_results.csv" in files:
            file_path = os.path.join(root, "loocv_results.csv")
            
            # Compute the relative path from the input folder
            rel_path = os.path.normpath(os.path.relpath(file_path, input_folder))
            rel_parts = rel_path.split(os.sep)
            
            # Ensure there are at least three folder levels: [..., segment_interval, independent_variable, file]
            if len(rel_parts) < 3:
                continue
            
            # Extract the folder names:
            # - Two folders up (index -3) for Segment Interval
            # - One folder up (index -2) for independent variable
            segment_interval_folder = rel_parts[-3]
            independent_variable_folder = rel_parts[-2]
            
            # Process folder names by splitting on whitespace and taking all tokens except the first one.
            seg_tokens = segment_interval_folder.split()
            segment_interval = " ".join(seg_tokens[1:]) if len(seg_tokens) > 1 else segment_interval_folder
            
            ind_tokens = independent_variable_folder.split()
            independent_variable = " ".join(ind_tokens[1:]) if len(ind_tokens) > 1 else independent_variable_folder
            watershed = ind_tokens[0] if len(ind_tokens) > 0 else "Unknown"
            # The Region is taken as the base name of the input folder.
            region = os.path.basename(os.path.normpath(input_folder))
            
            try:
                # Read the CSV and extract the first row
                df = pd.read_csv(file_path)
                if df.empty:
                    continue
                first_row = df.iloc[0]
                
                # Select the desired columns
                selected_columns = ["stats.bias", "stats.MSPE", "stats.RMSPE", "stats.RAV"]
                # If any column is missing, an error will be raised.
                selected_data = first_row[selected_columns]
                
                # Build a dictionary for this row
                row_data = {
                    "Region": region,
                    "Watershed": watershed,
                    "Segment Interval": segment_interval,
                    "independent variable": independent_variable,
                    "stats.bias": selected_data["stats.bias"],
                    "stats.MSPE": selected_data["stats.MSPE"],
                    "stats.RMSPE": selected_data["stats.RMSPE"],
                    "stats.RAV": selected_data["stats.RAV"]
                }
                summary_rows.append(row_data)
            except Exception as e:
                print(f"Error processing file {file_path}: {e}")

# Create a DataFrame from the collected rows and write to CSV
if summary_rows:
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(output_file, index=False)
    print(f"Summary saved to {output_file}")
else:
    print("No valid files found.")


Summary saved to Results Summary\loocv_summary.csv


## Aggregate glance, influence, tidy, and varcomp results to a single csv file

In [ ]:
import os
import pandas as pd

# List of input folders to search.
input_folders = ["CPF", "ETF"]  

# Mapping of target result file names to their summary file names.
target_files = {
    "glance_results.csv": r"Results Summary\glance_summary.csv",
    "influence_results.csv": r"Results Summary\influence_summary.csv",
    "tidy_results.csv": r"Results Summary\tidy_summary.csv",
    "varcomp_results.csv": r"Results Summary\varcomp_summary.csv"
}


# Initialize a dictionary to hold lists of DataFrames for each result type.
summary_data = { key: [] for key in target_files.keys() }

for base_folder in input_folders:
    # Walk through the directory tree.
    for root, dirs, files in os.walk(base_folder):
        # Skip subdirectories named "Archive"
        if "Archive" in dirs:
            dirs.remove("Archive")
            
        # Process each file in the current directory.
        for file in files:
            if file in target_files:
                filepath = os.path.join(root, file)
                print(f"Processing file: {filepath}")
                
                # Try to read the CSV file.
                try:
                    df = pd.read_csv(filepath)
                except Exception as e:
                    print(f"Error reading {filepath}: {e}")
                    continue


                # Compute metadata using the relative path from the base folder.
                relative_path = os.path.relpath(filepath, start=base_folder)
                parts = relative_path.split(os.path.sep)
                
                # 'Region' is defined as the name of the input folder (base folder).
                region = os.path.basename(os.path.normpath(base_folder))
                
                # For the following, we expect the file to be at least three levels deep.
                # In the example: "CPF/Outputs/Segmented 5m/CPF lidar erosion/glance_results.csv":
                #   - parts[-3] is "Segmented 5m" (Segment Interval),
                #   - parts[-2] is "CPF lidar erosion" (independent variable).
                if len(parts) >= 3:
                    seg_interval_full = parts[-3]
                    indep_var_full = parts[-2]
                else:
                    seg_interval_full = ""
                    indep_var_full = ""
                
                # For "Segment Interval": split the folder name by whitespace and take from the second token onward.
                seg_interval_tokens = seg_interval_full.split()
                segment_interval = " ".join(seg_interval_tokens[1:]) if len(seg_interval_tokens) > 1 else seg_interval_full
                
                # For "independent variable": do the same.
                indep_var_tokens = indep_var_full.split()
                independent_variable = " ".join(indep_var_tokens[1:]) if len(indep_var_tokens) > 1 else indep_var_full
                watershed = "".join(indep_var_tokens[0]) if len(indep_var_tokens) > 1 else indep_var_full
                # Insert the metadata columns at the beginning of the DataFrame.
                # (They will be added as new columns to the row from the CSV file.)
                df.insert(0, "Region", region)
                df.insert(1, "Watershed", watershed)
                df.insert(2, "Segment Interval", segment_interval)
                df.insert(3, "independent variable", independent_variable)
                # Optionally, include the source file path for troubleshooting.
                df["source_file"] = filepath
                
                # Append the resulting row to the corresponding summary list.
                summary_data[file].append(df)

# Write out each summary CSV file.
for result_filename, df_list in summary_data.items():
    if df_list:
        combined_df = pd.concat(df_list, ignore_index=True)
        # Remove 
        
        summary_filename = target_files[result_filename]
        combined_df.to_csv(summary_filename, index=False)
        print(f"Created summary file: {summary_filename} with {len(combined_df)} row(s).")
    else:
        print(f"No files found for {result_filename}.")



## Aggregate Model Performance Evaluation Files